# Storm attributes with zagg

This notebook demonstrates offloading the heavy per-storm MERRA-2 attribute
aggregation to [zagg](https://github.com/englacial/zagg)'s temporal pipeline
(design thread: [englacial/zagg#213](https://github.com/englacial/zagg/issues/213);
consumer contract: zagg's `docs/temporal_events.md`). artools stays a *declarative*
consumer: it exports per-storm events, resolves granules, and post-processes;
zagg streams the reductions — locally here, or one AWS Lambda invoke per storm at scale.

**Requirements:** `pip install -e .[zagg]`, an Earthdata login (`~/.netrc`), and a
local clone of the AR catalogs dataset (defaults to `../antarctic_AR_catalogs/`).


In [ ]:
from pathlib import Path

import pandas as pd
import zagg
from zagg.config import load_config

from artools.zagg_events import export_events
from artools.zagg_granules import build_granule_index, local_event_tuples
from artools.zagg_statics import prepare_static_files, load_zagg_statics
from artools.zagg_geometry import geometry_attributes, assemble_attribute_table

CATALOGS_DIR = Path('../antarctic_AR_catalogs')
CATALOG = CATALOGS_DIR / 'epsspace0.5_epstime12_minpts5_nreppts10_seed12345.h5'
FIELD_COLLECTIONS = ['merra2_slv', 'merra2_flx', 'merra2_v850', 'merra2_omega']


## 1. Export storms as zagg events

One monolithic catalog HDF5 → per-storm single-variable NetCDF masks (raw and
24-hour augmented) plus an event-catalog parquet. We demo on five storms: four
landfalling, one that never reaches the ice sheet (its landfall attributes come
back NaN by construction).


In [ ]:
full_catalog = pd.read_hdf(CATALOG)
storm_ids = [int(i) for i in list(full_catalog.index[full_catalog['is_landfalling']][:4])
             + list(full_catalog.index[~full_catalog['is_landfalling']][:1])]
events, parquet_path = export_events(CATALOG, storms=storm_ids)
events[['event_key', 'is_landfalling', 't_start', 't_end', 'n_timesteps']]


## 2. Statics and granule resolution

The AIS mask and cell areas normalize from files shipped with the catalogs
dataset. The monthly climatology (the baseline the two anomaly attributes
subtract) is built once from the MERRA-2 monthly means — **the baseline
period is a science choice**: e.g. the WMO 1991–2020 normal, or the full
1980–present record. Build it before anything else that consumes it (in
particular before mirroring statics to S3 for a Lambda run — the scale
runner staleness guard will force a re-mirror if the file changes
afterwards). A multi-decade build takes ~15–20 minutes.

The granule index is one CMR query per collection DOI, cached as JSON with
both the S3 (in-region/Lambda) and HTTPS (local streaming) link forms.


In [ ]:
prepare_static_files(CATALOGS_DIR)

# Build (or rebuild) the anomaly baseline -- run once, before any S3 mirror.
# Pick the baseline period deliberately; two common choices:
#   ('1991-01-01', '2020-12-31')   # WMO climate normal
#   ('1980-01-01', '2024-12-31')   # full MERRA-2 record
BUILD_CLIMATOLOGY = False  # flip once; ~15-20 min for a multi-decade baseline
if BUILD_CLIMATOLOGY:
    from artools.zagg_statics import build_climatology
    build_climatology(('1991-01-01', '2020-12-31'),
                      CATALOGS_DIR / 'static' / 'merra2_monthly_climatology.nc')

statics = load_zagg_statics(CATALOGS_DIR)

span = (str(pd.Timestamp(events['t_start'].min()).date()),
        str(pd.Timestamp(events['t_end_augmented'].max()).date()))
index = build_granule_index(span, cache_path=CATALOGS_DIR / 'derived' / 'granule_index.json')


## 3. Run zagg locally

Two runs, because the precipitation attributes aggregate under a *different*
event set (the 24-hour augmented masks): `configs/ar_attributes.yaml` under the
raw masks, `configs/ar_precip.yaml` under the augmented ones. All aggregation
semantics — masks, triggers, anomaly transform, per-collection time alignment —
live in the YAML; nothing here is storm-specific code.


In [ ]:
raw_tuples = local_event_tuples(events, index, FIELD_COLLECTIONS, statics)
fields = zagg.agg(load_config('configs/ar_attributes.yaml'), events=raw_tuples)
pd.DataFrame([{'event_key': r['event_key'], **r['results']} for r in fields['results']])


In [ ]:
aug_tuples = local_event_tuples(events, index, ['merra2_precip'], statics, augmented=True)
precip = zagg.agg(load_config('configs/ar_precip.yaml'), events=aug_tuples)
pd.DataFrame([{'event_key': r['event_key'], **r['results']} for r in precip['results']])


## 4. Geometry attributes and the final table

The mask-geometry attributes never stream MERRA-2, so they stay client-side —
the same `attribute_utils` kernels the reference pipeline uses — and everything
joins on `event_key`.


In [ ]:
geometry = geometry_attributes(events, statics)
fields_df = pd.DataFrame([{'event_key': r['event_key'], **r['results']} for r in fields['results']])
precip_df = pd.DataFrame([{'event_key': r['event_key'], **r['results']} for r in precip['results']])
table = assemble_attribute_table(events, geometry, [fields_df, precip_df])
table


## 5. Parity against the reference kernels

`scripts/zagg_parity.py` runs the same storms through BOTH paths — this zagg
pipeline and the original `attribute_utils` kernels — over the same opened
datasets and prints a per-attribute comparison:

```bash
python scripts/zagg_parity.py --n-storms 6
```


## 6. Scaling out on AWS Lambda (operator-run)

The full catalog is 8,714 storms; the fan-out is one `process_event` Lambda
invoke per storm. Running it requires a zagg serverless backend standing in
**your own AWS account** (region `us-west-2` — GES DISC S3 is in-region only).
The zagg documentation covers enabling that end to end:

- **Required:** [Standing up the backend](https://docs.englacial.org/zagg/deployment/standup/) —
  one CloudFormation command (`stand_up.sh`) creates the execution role, the
  dependency layer, and the `process-shard` function from release artifacts.
- *Informational:* [AWS Lambda details](https://docs.englacial.org/zagg/deployment/lambda/) —
  layer/function anatomy, concurrency, and the manual deploy path.
- *Informational:* [Execution role](https://docs.englacial.org/zagg/deployment/execution-role/) —
  the IAM permissions the workers need.

Environment: the runner needs this repo installed with the zagg extra
(`uv venv && uv pip install -e ".[zagg]"`, then use `.venv/bin/python`), an
Earthdata login (`~/.netrc`), and the **production climatology built first**
(section 2) so the mirrored baseline is the one the science calls for.

Once the stack is up, `scripts/zagg_scale_run.py` in this repo is the maintained
runner (mirrors inputs to your bucket, dispatches both attribute runs, collects
the Parquet outputs — default is a 10-storm slice; `--full` for the catalog):

```bash
env AWS_PROFILE=... python scripts/zagg_scale_run.py \
    --s3-prefix s3://YOUR-BUCKET/ar-events/eps0.5-seed12345
```

The cell below shows the same flow inline for reference. **It spends real
money and requires the stack above; leave `RUN_LAMBDA` False unless you are
the operator.**


In [ ]:
RUN_LAMBDA = False
if RUN_LAMBDA:
    from artools.zagg_granules import events_for_zagg, mirror_to_s3
    all_events, _ = export_events(CATALOG)  # every storm
    static_paths = {
        'ais_mask': CATALOGS_DIR / 'static' / 'ais_mask.nc',
        'cell_areas': CATALOGS_DIR / 'static' / 'cell_areas.nc',
        'climatology': CATALOGS_DIR / 'static' / 'merra2_monthly_climatology.nc',
    }
    uri_map, static_uris = mirror_to_s3(
        all_events,
        static_paths,
        's3://YOUR-BUCKET/ar-events/eps0.5-seed12345/')
    payloads = events_for_zagg(all_events, index, FIELD_COLLECTIONS,
                               static_uris, uri_map=uri_map.get)
    fields = zagg.agg(load_config('configs/ar_attributes.yaml'),
                      events=payloads, backend='lambda',
                      store='s3://YOUR-BUCKET/ar-output/ar_attributes.parquet')